# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS42024/Shailesh-Flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [20]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
MARKER = "data/raw/content_refresh_anonymized.csv"

def find_repo_root(max_up=6):
    here = Path.cwd()
    for base in [here, *here.parents[:max_up]]:
        if (base / MARKER).exists():
            return base
        for child in base.glob("*/"):
            if (child / MARKER).exists():
                return child
    return None

def clone_repo(into: Path):
    print(f"Repo not found nearby -- cloning it into: {into}")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(into)], check=True)

def pull_repo(at: Path):
    print(f"Existing checkout at {at} looks incomplete -- trying git pull...")
    subprocess.run(["git", "-C", str(at), "pull"], check=True)

def ensure_repo(into: Path):
    if not into.exists():
        clone_repo(into)
    elif not (into / MARKER).exists():
        try:
            pull_repo(into)
        except subprocess.CalledProcessError:
            print("git pull failed -- checkout may be broken. Consider deleting it and rerunning.")

def install_requirements(at: Path):
    req = at / "requirements.txt"
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
    else:
        print(f"No requirements.txt found at {req}, skipping install.")

root = None
try:
    if IN_COLAB:
        target = Path(REPO_DIR)
        ensure_repo(target)
        os.chdir(target)
        root = Path.cwd()
        install_requirements(root)
    else:
        root = find_repo_root()
        if root is None:
            target = Path.cwd() / REPO_DIR
            ensure_repo(target)
            root = target
        os.chdir(root)
        install_requirements(root)
except (subprocess.CalledProcessError, FileNotFoundError) as e:
    print(f"Setup failed ({e}). Do you have git installed and internet access?")
    root = None

print("Working dir:", os.getcwd())
csv_ok = os.path.exists(MARKER)
if csv_ok:
    print("Starter data found. You're ready.")
else:
    print("Starter CSV still not found. Contents here:", os.listdir("."))

Repo not found nearby -- cloning it into: flyrank-ml-internship-starter
Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [21]:
#I'll predict is_declining_label (trend_direction == "down"), which is a defined rule calculated from the current window — not an observed future outcome — so it's a proxy label, not a true target.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# How the proxy label splits
label_counts = df["trend_direction"].value_counts()
print("trend_direction value counts:")
print(label_counts)
print(f"\n'down' share (my positive label): {(df['trend_direction'] == 'down').mean():.1%}")

# Sanity check: is this rule calculated from the SAME window as the features?
print(f"\nMedian content_age_days: {df['content_age_days'].median():.0f}")
print(f"Median impressions_90d: {df['impressions_90d'].median():.0f}")


trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

'down' share (my positive label): 54.2%

Median content_age_days: 236
Median impressions_90d: 731


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [18]:

#I'll Use The Precision@50 - of the top 50 pages my model ranks first, how many are actually labeled 'declining' - because the baseline rule scores 0.240 on this metric and I'd call anything meaningfully above that.

import json

with open("outputs/model_results.json") as f:
    results = json.load(f)

# Baseline metrics live at the top level, with a "baseline_" prefix
baseline_p50 = results["baseline"]["baseline_precision_at_50"]
print(f"Baseline precision@50: {baseline_p50:.3f} (~{baseline_p50*50:.0f} of top 50 correct)")

print()
# The three trained models live inside "models", keyed by name
for name, m in results["models"].items():
    print(f"{name:20s} Precision@50 = {m['precision_at_50']:.3f}")

# "best_model" tells you which one won, by name
best_name = results["best_model"]["name"]
best_p50 = results["models"][best_name]["precision_at_50"]
print()
print(f"Best model ('{best_name}') precision@50: {best_p50:.3f} "
      f"(~{best_p50*50:.0f} of top 50 correct)")
print(f"Improvement over baseline: +{(best_p50 - baseline_p50):.3f}")

Baseline precision@50: 0.240 (~12 of top 50 correct)

decision_tree        Precision@50 = 0.580
logistic_regression  Precision@50 = 0.400
random_forest        Precision@50 = 0.740

Best model ('random_forest') precision@50: 0.740 (~37 of top 50 correct)
Improvement over baseline: +0.500


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [25]:
#One row in content_refresh_anonymized.csv = one content page (content_id), with its search and engagement metrics rolled up over the same trailing 90-day window — confirmed below, since all 30,000 rows have a unique content_id.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total rows:", len(df))
print("Unique content_id values:", df["content_id"].nunique())
print("Are content_id values unique (one row per page)?", df["content_id"].is_unique)

print()
print("One real row, transposed so it reads top-to-bottom:")
print(df.iloc[0])

Total rows: 30000
Unique content_id values: 30000
Are content_id values unique (one row per page)? True

One real row, transposed so it reads top-to-bottom:
content_id                content_304f48230142
client_id                    client_f369cb89fc
search_volume                             10.0
competition                               0.67
competition_level                         HIGH
cpc                                       2.05
content_type                   keyword article
main_intent                      transactional
word_count                              3221.0
char_count                             20457.0
provider_used                              NaN
model_used                    gemini-2.5-flash
impressions_90d                           3803
clicks_90d                                  29
pageviews_90d                               22
sessions_90d                                17
users_90d                                   16
engaged_sessions_90d                        

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [30]:

#A fixed if-statement rule applies the same hand-picked weights to every page, but the signals interact differently case by case — one page has a great position (3.4) and CTR (0.89) yet the baseline still ranks it as urgent priority #19 even though it isn't declining, while another page with weak position (26.9) and CTR (0.09) is buried at baseline rank 8375 despite genuinely declining — and a fixed formula can't tell these two patterns apart the way a model trained on the interactions between all 52 features can.


import os, subprocess

if not (os.path.exists("data/processed/baseline_refresh_queue.csv")
        and os.path.exists("data/processed/model_predictions.csv")):
    print("Processed files not found — running the pipeline first...")
    subprocess.run(["python", "scripts/run_all.py"], check=True)

import pandas as pd
base = pd.read_csv("data/processed/baseline_refresh_queue.csv")
preds = pd.read_csv("data/processed/model_predictions.csv")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.